# Masking Policies for Sensitive Columns

Applies dynamic data masking to all columns classified as sensitive (IDENTIFIER, QUASI_IDENTIFIER, SENSITIVE) by `SYSTEM$CLASSIFY`.

| Privacy Category | Masking Strategy |
|---|---|
| IDENTIFIER | Fully masked → `***MASKED***` |
| QUASI_IDENTIFIER | Partially masked → first 2 chars + `****` |
| SENSITIVE | Nullified → `0` for numbers |

In [ ]:
%%sql -r set_role
USE ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r create_schema
CREATE SCHEMA IF NOT EXISTS GOVERNANCE_DB.MASKING;

In [ ]:
%%sql -r policy_text_id
CREATE OR REPLACE MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_IDENTIFIER AS
  (val STRING) RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN') THEN val
    ELSE '***MASKED***'
  END;

In [ ]:
%%sql -r policy_text_quasi
CREATE OR REPLACE MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_QUASI_IDENTIFIER AS
  (val STRING) RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN') THEN val
    ELSE CONCAT(LEFT(val, 2), '****')
  END;

In [ ]:
%%sql -r policy_number
CREATE OR REPLACE MASKING POLICY GOVERNANCE_DB.MASKING.MASK_NUMBER_SENSITIVE AS
  (val NUMBER) RETURNS NUMBER ->
  CASE
    WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN') THEN val
    ELSE 0
  END;

In [ ]:
%%sql -r policy_array
CREATE OR REPLACE MASKING POLICY GOVERNANCE_DB.MASKING.MASK_ARRAY_QUASI_IDENTIFIER AS
  (val ARRAY) RETURNS ARRAY ->
  CASE
    WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN') THEN val
    ELSE ARRAY_CONSTRUCT('***MASKED***')
  END;

In [ ]:
%%sql -r apply_identifier
ALTER TABLE CALL_CENTER_DB.RAW.AUDIO_FILES
  MODIFY COLUMN AUDIO_FILE_URL SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_IDENTIFIER;

ALTER TABLE CALL_CENTER_DB.STAGE.TRANSCRIBE_AUDIO_FILES
  MODIFY COLUMN AUDIO_FILE_URL SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_IDENTIFIER;

ALTER TABLE FINOPS.ANALYTICS.DIM_USER
  MODIFY COLUMN EMAIL SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_IDENTIFIER;

ALTER TABLE FINOPS.ANALYTICS_ANALYTICS.DIM_USER
  MODIFY COLUMN EMAIL SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_IDENTIFIER;

ALTER TABLE TEST_DB.PUBLIC.EMPLOYEE
  MODIFY COLUMN ENAME SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_IDENTIFIER;

ALTER TABLE TEST_DB.PUBLIC.RLS_MAPPING
  MODIFY COLUMN ENAME SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_IDENTIFIER;

In [ ]:
%%sql -r apply_quasi
ALTER TABLE CALL_CENTER_DB.IMAGES.CORTEX_AI_USAGE_AUDIT
  MODIFY COLUMN ROLE_NAME SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_QUASI_IDENTIFIER;

ALTER TABLE FINOPS.ANALYTICS.DIM_USER
  MODIFY COLUMN DEFAULT_ROLE SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_QUASI_IDENTIFIER;

ALTER TABLE FINOPS.ANALYTICS.FACT_CREDITS
  MODIFY COLUMN ROLE_NAME SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_QUASI_IDENTIFIER;

ALTER TABLE FINOPS.ANALYTICS_ANALYTICS.DIM_USER
  MODIFY COLUMN DEFAULT_ROLE SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_QUASI_IDENTIFIER;

ALTER TABLE FINOPS.ANALYTICS_ANALYTICS.FACT_CREDITS
  MODIFY COLUMN ROLE_NAME SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_QUASI_IDENTIFIER;

ALTER TABLE INTELLIGENCE_DB.RETAIL.SUPPORT_CASES
  MODIFY COLUMN TITLE SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_QUASI_IDENTIFIER;

ALTER TABLE TEST_DB.PUBLIC.DAILY_CREDIT_SPENDING
  MODIFY COLUMN ROLE_NAME SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_QUASI_IDENTIFIER;

ALTER TABLE TEST_DB.PUBLIC.EMPLOYEE
  MODIFY COLUMN POSTALCODE SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_TEXT_QUASI_IDENTIFIER;

In [ ]:
%%sql -r apply_sensitive
ALTER TABLE TEST_DB.PUBLIC.EMPLOYEE
  MODIFY COLUMN SALARY SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_NUMBER_SENSITIVE;

In [ ]:
%%sql -r apply_array
ALTER TABLE FINOPS.ANALYTICS.DIM_USER
  MODIFY COLUMN ASSIGNED_ROLES SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_ARRAY_QUASI_IDENTIFIER;

ALTER TABLE FINOPS.ANALYTICS_ANALYTICS.DIM_USER
  MODIFY COLUMN ASSIGNED_ROLES SET MASKING POLICY GOVERNANCE_DB.MASKING.MASK_ARRAY_QUASI_IDENTIFIER;

In [ ]:
%%sql -r show_policies
SHOW MASKING POLICIES IN SCHEMA GOVERNANCE_DB.MASKING;

In [ ]:
%%sql -r policy_refs
SELECT POLICY_NAME, REF_DATABASE_NAME, REF_SCHEMA_NAME, REF_ENTITY_NAME, REF_COLUMN_NAME
FROM TABLE(INFORMATION_SCHEMA.POLICY_REFERENCES(POLICY_NAME => 'GOVERNANCE_DB.MASKING.MASK_TEXT_IDENTIFIER'));

In [ ]:
%%sql -r test_masked
SELECT ENAME, SALARY, POSTALCODE FROM TEST_DB.PUBLIC.EMPLOYEE LIMIT 5;